教学演示 思维链

**本课两个重点**：
1. 用 **LangChain `PromptTemplate`** 生成与 `str.format(task=…)` 相同的完整提示词，并**展示全文**（教学演示「思维链提示词长什么样」）。
2. （可选）用 **`ChatOpenAI`** 发一条请求，看模型按四段标题（GIVEN / LOOK UP / DERIVE / GUESSES）回复。


## 0. 环境

```bash
pip install langchain-core langchain-openai httpx
```

若最后一格要真调模型，请设置：`OPENAI_API_KEY`、`OPENAI_BASE_URL`、`OPENAI_MODEL`（或在代码里临时写死）。

In [1]:
# Windows 下避免 print 长中文/特殊符号时报编码错（可选）
import sys

if sys.platform == "win32":
    for _s in (sys.stdout, sys.stderr):
        try:
            _s.reconfigure(encoding="utf-8")
        except Exception:
            pass

## 1. 思维链提示词模板

占位符只有一个：`{task}` → 替换为用户的自然语言任务。

In [3]:
FACTS_PROMPT = """Below I will present you a request. Before we begin addressing the request, please answer the following pre-survey to the best of your ability. Keep in mind that you are Ken Jennings-level with trivia, and Mensa-level with puzzles, so there should be a deep well to draw from.

Here is the request:

{task}

Here is the pre-survey:

    1. Please list any specific facts or figures that are GIVEN in the request itself. It is possible that there are none.
    2. Please list any facts that may need to be looked up, and WHERE SPECIFICALLY they might be found. In some cases, authoritative sources are mentioned in the request itself.
    3. Please list any facts that may need to be derived (e.g., via logical deduction, simulation, or computation)
    4. Please list any facts that are recalled from memory, hunches, well-reasoned guesses, etc.

When answering this survey, keep in mind that "facts" will typically be specific names, dates, statistics, etc. Your answer should use headings:

    1. GIVEN OR VERIFIED FACTS
    2. FACTS TO LOOK UP
    3. FACTS TO DERIVE
    4. EDUCATED GUESSES

DO NOT include any other headings or sections in your response. DO NOT list next steps or plans until asked to do so.
"""

# 课堂示例任务
task = """What is the UV index in Melbourne today?"""

## 2. 用 LangChain 生成「完整提示词」

`PromptTemplate.from_template` 与 Python 的 `FACTS_PROMPT.format(task=…)` 在这里**等价**，便于后面接 LCEL、链式调试等。

In [4]:
from langchain_core.prompts import PromptTemplate

facts_template = PromptTemplate.from_template(FACTS_PROMPT)
full_prompt = facts_template.format(task=task.strip())

# 与纯 format 一致（可自行对照）
assert full_prompt == FACTS_PROMPT.format(task=task.strip())

C:\Users\zhanghong26\.conda\envs\enterprise_bench_agents_test\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. 展示：拼好后的整条「思维链」提示词

下面就是编排器第一步要发给模型的**完整用户消息**（节选可在课堂改成 `[:2000]` 等）。

In [5]:
from IPython.display import display, Markdown

display(Markdown(f"**字符数：** {len(full_prompt)}"))
display(Markdown("```text\n" + full_prompt + "\n```"))

**字符数：** 1248

```text
Below I will present you a request. Before we begin addressing the request, please answer the following pre-survey to the best of your ability. Keep in mind that you are Ken Jennings-level with trivia, and Mensa-level with puzzles, so there should be a deep well to draw from.

Here is the request:

What is the UV index in Melbourne today?

Here is the pre-survey:

    1. Please list any specific facts or figures that are GIVEN in the request itself. It is possible that there are none.
    2. Please list any facts that may need to be looked up, and WHERE SPECIFICALLY they might be found. In some cases, authoritative sources are mentioned in the request itself.
    3. Please list any facts that may need to be derived (e.g., via logical deduction, simulation, or computation)
    4. Please list any facts that are recalled from memory, hunches, well-reasoned guesses, etc.

When answering this survey, keep in mind that "facts" will typically be specific names, dates, statistics, etc. Your answer should use headings:

    1. GIVEN OR VERIFIED FACTS
    2. FACTS TO LOOK UP
    3. FACTS TO DERIVE
    4. EDUCATED GUESSES

DO NOT include any other headings or sections in your response. DO NOT list next steps or plans until asked to do so.

```

## 4.（可选）LangChain `ChatOpenAI` 调用一次

把上格生成的 `full_prompt` 作为**一条用户消息**发给 OpenAI 兼容网关即可。下面用环境变量；课堂演示也可改成与你 `ChatOpenAI(model=..., base_url=..., api_key=...)` 一样的字面量。

In [6]:
import os

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

kw = dict(
        model="gpt-oss-120b",
        temperature=0,
        base_url="https://modelfactory.lenovo.com:28080/api",
        api_key="your api key ",
)
import httpx
kw["http_client"] = httpx.Client(verify=False)
llm = ChatOpenAI(**kw)
resp = llm.invoke([HumanMessage(content=full_prompt)])
display(Markdown("### 模型输出"))
display(Markdown(resp.content if isinstance(resp.content, str) else str(resp.content)))

### 模型输出

**1. GIVEN OR VERIFIED FACTS**
- The request asks for the UV index in **Melbourne** “today.”
- The current date is **2026‑05‑19** (as indicated by the system).

**2. FACTS TO LOOK UP**
- The **current UV index** (or forecasted UV index for the day) for Melbourne on 2026‑05‑19.  
  *Potential sources:*  
  • Australian Bureau of Meteorology (BOM) – “UV Index” page for Melbourne.  
  • Environment Australia’s “Bureau of Meteorology – UV Index” API.  
  • Weather.com (The Weather Channel) – Melbourne UV Index today.  
  • MetService or AccuWeather – Melbourne UV forecast.  
  • Local government health department UV alerts (e.g., Victorian Health).

**3. FACTS TO DERIVE**
- If only a UV forecast range is provided (e.g., “moderate (3‑5)”), derive the specific numeric value that would be reported as “the UV index” (typically the maximum predicted value for the day).  
- Convert any time‑specific UV readings (e.g., peak at 12 pm) to a single “today’s UV index” figure.

**4. EDUCATED GUESSES**
- Melbourne in mid‑May (autumn) usually experiences **low to moderate** UV levels. Historical data suggest typical daily UV indices of **1–3**, occasionally reaching **4** on clear sunny days.  
- Therefore, a reasonable guess for the UV index in Melbourne on 2026‑05‑19 would be **around 2** (low) with a possible range of **1–4** depending on cloud cover.

## 5. 小结

- **思维链提示词**在这里 = 固定英文说明 + 嵌入的 `{task}` + 要求模型按四段标题作答；**全部放在一条 user 内容里**。
- **LangChain**：`PromptTemplate` 负责字符串拼装；`ChatOpenAI` 负责调用；